<a href="https://www.kaggle.com/code/yamanbashar/notebook936007e0f2?scriptVersionId=325605466" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
!pip install ultralytics
!pip install pycocotools

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
from pathlib import Path
import yaml
import json
from pycocotools.coco import COCO
import random
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import copy
import os
import torch
from ultralytics import YOLO
from ultralytics.data.dataset import DATASET_CACHE_VERSION, YOLODataset
from ultralytics.data.utils import get_hash, load_dataset_cache_file, save_dataset_cache_file
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils import TQDM, colorstr
from ultralytics.data.converter import convert_coco
from torchvision import transforms, datasets, models
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import torchvision
from PIL import Image
from bs4 import BeautifulSoup
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
print(torch.cuda.device_count())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class CONFIG:
    MAX_IMAGES = 5000
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.7
    TEST_RATIO = 0.3
    RANDOM_SEED = 42
    IMAGE_DIR = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017"
    INPUT = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/instances_train2017.json"
    OUTPUT =  "/kaggle/working/instances_5class_train.json"
    CLASSES = ['person', "car", "bicycle", "dog", "bus"]

## Preprocessing

In [ ]:
coco = COCO(CONFIG.INPUT) #load coco

cat_ids = coco.getCatIds(catNms=CONFIG.CLASSES) #get category ids
img_ids = set()
for cat_id in cat_ids:
    ids = coco.getImgIds(catIds=[cat_id])
    img_ids.update(ids)
img_ids = list(img_ids)
print(len(img_ids))

In [ ]:
cat_id_set = set(cat_ids)
img_id_set = set(img_ids)
print(cat_ids, cat_id_set)
print(len(img_id_set))

ann_ids = coco.getAnnIds(catIds=cat_ids)
anns = coco.loadAnns(ann_ids)
print(Counter([ann["category_id"] for ann in anns]))
##Essentially, we filter the annotations that we are going to use here to make sure that 
#when we load the images we wont load annotations we dont want to use
filtered_anns = []
for ann in anns:
    if ann["category_id"] in cat_id_set and ann:
        filtered_anns.append(ann) 

filtered_img_ids = set(a["image_id"] for a in filtered_anns)
filtered_img_ids = set(list(filtered_img_ids)[:CONFIG.MAX_IMAGES])
filtered_images = coco.loadImgs(list(filtered_img_ids))
print(f"Images: {len(filtered_images)}, Annotations: {len(filtered_anns)}")

#apply max images.
new_filtered_anns = []

for ann in filtered_anns:
    if ann["image_id"] in filtered_img_ids:
        new_filtered_anns.append(ann)

filtered_anns = new_filtered_anns 

id_map = {} #remapping class ids because yolo will need contiguous labels, cant have the labels be ex: [1, 5, 15, 20] 
for new_id, old_id in  enumerate(cat_ids):
    id_map[old_id] = new_id

filtered_anns = copy.deepcopy(filtered_anns)
for ann in filtered_anns:
    ann["category_id"] = id_map[ann["category_id"]]

new_coco = {
    "images": filtered_images,
    "annotations": filtered_anns,
    "categories": [
        {"id": new_id, "name": name}
        for new_id, name in enumerate(CONFIG.CLASSES)
    ]
}

#save the new annotation file
with open(CONFIG.OUTPUT, "w") as f:
    json.dump(new_coco, f)
print(f"Saved the filtered annotation file at {CONFIG.OUTPUT}")


In [ ]:
print("images:", len(img_ids))
print("cats:", cat_ids)
print("classes:", CONFIG.CLASSES)
ann_ids = coco.getAnnIds(imgIds=img_ids, catIds=cat_ids)
print("raw filtered annotations:", len(ann_ids))
anns = coco.loadAnns(ann_ids)
print("loaded anns:", len(anns))
print("unique classes:", len(set(a["category_id"] for a in anns)))
counts = Counter([a["category_id"] for a in anns])
print(counts)
img_used = set(a["image_id"] for a in anns)
print("images with labels:", len(img_used))
missing_imgs = set(img_ids) - img_used
print("images with NO selected classes:", len(missing_imgs))
img = coco.loadImgs(list(img_used)[0])[0]
print(img)
aid = coco.getAnnIds(imgIds=img["id"], catIds=cat_ids)
sample = coco.loadAnns(aid)
print(len(sample), sample[:2])

### Train Val Test Split

In [ ]:
new_categories = [
    {"id": i, "name": name} 
    for i, name in enumerate(CONFIG.CLASSES)
]

# First split off test
train_val_ids, test_ids = train_test_split(
    list(filtered_img_ids),
    test_size=CONFIG.TEST_RATIO,
    random_state=CONFIG.RANDOM_SEED,
    shuffle=True
)

train_ids, val_ids = train_test_split(
    train_val_ids,
    train_size=CONFIG.TRAIN_RATIO,
    random_state=CONFIG.RANDOM_SEED,
    shuffle=True
)

train_id_set = set(train_ids)
val_id_set = set(val_ids)
test_id_set  = set(test_ids)

train_anns = [a for a in filtered_anns if a["image_id"] in train_id_set]
val_anns   = [a for a in filtered_anns if a["image_id"] in val_id_set]
test_anns  = [a for a in filtered_anns if a["image_id"] in test_id_set]

train_images = coco.loadImgs(list(train_id_set))
val_images   = coco.loadImgs(list(val_id_set))
test_images  = coco.loadImgs(list(test_id_set))

train_coco = {
    "images": train_images,
    "annotations": train_anns,
    "categories": new_categories
}

val_coco = {
    "images": val_images,
    "annotations": val_anns,
    "categories": new_categories
}

test_coco = {
    "images": test_images,
    "annotations": test_anns,
    "categories": new_categories
}

for split_name, imgs, anns in [("train", train_images, train_anns), ("val", val_images, val_anns), ("test", test_images, test_anns)]:
    coco_dict = {"images": imgs, "annotations":anns, "categories": new_categories}
    with open(f"/kaggle/working/{split_name}.json", "w") as f:
        json.dump(coco_dict, f)
    print(f"{split_name}: {len(imgs)} images, {len(anns)} annotations")
print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")


### Convert to YOLO format

In [ ]:
class COCODataset(YOLODataset):
    def __init__(self, *args, json_file="", **kwargs):
        self.json_file = json_file
        super().__init__(*args, data={"channels": 3}, **kwargs)
    def get_img_files(self, img_path):
        return []
    def cache_labels(self, path=Path("./labels.cache")):
        x = {"labels": []}
        with open(self.json) as f:
            coco = json.load(f)
        categories = {cat["id"]: i for i in enumerate(sorted(coco["categories"], key = lambda c: c["id"]))}
        img_to_anns = defaultdict(list)
        for ann in coco["annotations"]:
            img_to_anns

In [ ]:
def coco_to_yolo(coco_dict, label_dir):
    os.makedirs(label_dir, exist_ok=True)
    img_lookup = {img["id"]: img for img in coco_dict["images"]}
    
    ann_by_img = defaultdict(list)
    for ann in coco_dict["annotations"]:
        ann_by_img[ann["image_id"]].append(ann)
    
    for img_id, img_info in img_lookup.items():
        W, H = img_info["width"], img_info["height"]
        label_path = os.path.join(label_dir, img_info["file_name"].replace(".jpg", ".txt"))
        with open(label_path, "w") as f:
            for ann in ann_by_img[img_id]:
                if ann.get("iscrowd", 0):
                    continue
                x, y, w, h = ann["bbox"]
                cx = (x + w / 2) / W
                cy = (y + h / 2) / H
                nw, nh = w / W, h / H
                f.write(f"{ann['category_id']} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")

coco_to_yolo(train_coco, "/kaggle/working/labels/train")
coco_to_yolo(val_coco,   "/kaggle/working/labels/val")
coco_to_yolo(test_coco,  "/kaggle/working/labels/test")
print("Done")

In [ ]:
for split, coco_dict in [("train", train_coco), ("val", val_coco), ("test", test_coco)]:
    os.makedirs(f"/kaggle/working/images/{split}", exist_ok=True)
    for img in coco_dict["images"]:
        src = os.path.join(CONFIG.IMAGE_DIR, img["file_name"])
        dst = f"/kaggle/working/images/{split}/{img['file_name']}"
        if not os.path.exists(dst):
            os.symlink(src, dst)
data_yaml = {
    "path": "/kaggle/working",
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc": len(CONFIG.CLASSES),
    "names": CONFIG.CLASSES
}
with open("/kaggle/working/dataset.yaml", "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

In [ ]:
import os

for split in ["train", "val", "test"]:
    path = f"/kaggle/working/images/{split}"
    print(split, "files:", len(os.listdir(path)))
import glob

train_files = glob.glob("/kaggle/working/images/train/*")

print("train sample:", train_files[:5])
print("count:", len(train_files))

from PIL import Image

img_path = train_files[0]
img = Image.open(img_path)
img.show()

import os

broken = 0
for f in train_files[:100]:
    if not os.path.exists(f):
        broken += 1

print("broken links:", broken)

print("train labels:", len(os.listdir("/kaggle/working/labels/train")))
print("val labels:", len(os.listdir("/kaggle/working/labels/val")))


for root, dirs, files in os.walk("/kaggle/working"):
    if len(files) > 0:
        print(root, "->", len(files))


print(len(os.listdir("/kaggle/working/labels/train")))
print(len(os.listdir("/kaggle/working/images/train")))

## Faster R-CNN


In [ ]:
class COCOSubsetDataset(object):
    def __init__(self, json_path, image_dir):
        with open(json_path) as f:
            data = json.load(f)
        self.img_dir = image_dir
        self.imgs = data['images']

        #build image_id to annotations lookup
        self.ann_lookup = {}
        for ann in data["annotations"]:
            img_id = ann["image_id"]
            if img_id not in self.ann_lookup:
                self.ann_lookup[img_id] = []
            self.ann_lookup[img_id].append(ann)
        self.transforms = transforms.Compose([transforms.ToTensor()])
    def __getitem__(self, idx):
        img_info = self.imgs[idx]
        img_id = img_info["id"]

        img = Image.open(os.path.join(self.img_dir, img_info["file_name"])).convert("RGB")

        anns = self.ann_lookup.get(img_id, [])
        boxes, labels = [], []

        for ann in anns:
            if ann.get("iscrowd", 0):
                continue
            x, y, w, h = ann["bbox"]
            boxes.append([x, y, x + w, y + h])
            labels.append(ann["category_id"] + 1)

        #make sure we dont have invalid annotated images
        if len(boxes) == 0:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        
        target = {
            "boxes" : boxes,
            "labels" : labels,
            "image_ids": torch.tensor([img_id])
        }

        return self.transforms(img), target
    def __len__(self):
        return len(self.imgs)
def collate_fn(batch):
    return tuple(zip(*batch))
def get_model(num_classes):

    # Load a pre-trained Faster R-CNN model
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    )

    # Get the number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # Replace the pre-trained classifier head
    model.roi_heads.box_predictor = FastRCNNPredictor(
        in_features,
        num_classes
    )

    return model

In [ ]:
from torch.utils.data import DataLoader

IMAGE_DIR  = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017"
TRAIN_JSON = "/kaggle/working/train.json"
VAL_JSON   = "/kaggle/working/val.json"
TEST_JSON  = "/kaggle/working/test.json"
CLASSES    = ['person', 'car', 'bicycle', 'dog', 'bus']

trainset = COCOSubsetDataset(TRAIN_JSON, IMAGE_DIR)
valset = COCOSubsetDataset(VAL_JSON, IMAGE_DIR)
testset = COCOSubsetDataset(TEST_JSON, IMAGE_DIR)

train_loader = DataLoader(trainset, batch_size=4,  shuffle=True,  collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader   = DataLoader(valset,   batch_size=4,  shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=1,  shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)

In [ ]:
EPOCHS = 20

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm


CONFIGS = [
    {"lr": 1e-3, "momentum": 0.9,  "weight_decay": 1e-5},
    {"lr": 5e-4, "momentum": 0.95, "weight_decay": 1e-4},
    {"lr": 1e-2, "momentum": 0.9,  "weight_decay": 1e-3},
]

all_results = []

for cfg_idx, cfg in enumerate(CONFIGS):
    print(f"\n{'='*50}")
    print(f"Config {cfg_idx+1}/3: {cfg}")
    print(f"{'='*50}")

    model = get_model(num_classes=6)
    model.to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=cfg["lr"],
        momentum=cfg["momentum"],
        weight_decay=cfg["weight_decay"]
    )
    scaler = GradScaler()

    config_results = {
        "config": cfg,
        "epoch_losses": [],
        "epoch_val_losses": [],
        "epoch_map50": [],
        "epoch_map50_95": [],
    }

    for epoch in range(EPOCHS):
        # ---- TRAIN ----
        model.train()
        epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"[Config {cfg_idx+1}] Epoch {epoch+1}/{EPOCHS}", leave=True)

        for imgs, annotations in pbar:
            imgs = [img.to(device) for img in imgs]
            annotations = [{k: v.to(device) for k, v in t.items()} for t in annotations]

            optimizer.zero_grad()
            with autocast():
                loss_dict = model(imgs, annotations)
                losses = sum(loss for loss in loss_dict.values())

            scaler.scale(losses).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_value = losses.item()
            epoch_loss += loss_value
            pbar.set_postfix(loss=f"{loss_value:.4f}")

        avg_loss = epoch_loss / len(train_loader)
        config_results["epoch_losses"].append(avg_loss)

        # ---- VALIDATE ----
        model.eval()
        metric = MeanAveragePrecision(iou_type="bbox")
        
        # val loss + mAP in one pass
        val_epoch_loss = 0
        with torch.no_grad():
            for imgs, annotations in tqdm(val_loader, desc="  Validating", leave=False):
                imgs = [img.to(device) for img in imgs]
                annotations = [{k: v.to(device) for k, v in t.items()} for t in annotations]
        
                # get val loss (needs train mode)
                model.train()
                with torch.amp.autocast("cuda"):
                    loss_dict = model(imgs, annotations)
                    val_epoch_loss += sum(v.item() for v in loss_dict.values())
                model.eval()
        
                # get predictions for mAP
                with torch.amp.autocast("cuda"):
                    preds = model(imgs)
        
                preds_cpu = [{k: v.cpu() for k, v in p.items()} for p in preds]
                targets_cpu = [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in annotations]
                metric.update(preds_cpu, targets_cpu)
        
        avg_val_loss = val_epoch_loss / len(val_loader)
        map_scores = metric.compute()
        map50 = map_scores["map_50"].item()
        map50_95 = map_scores["map"].item()
        
        config_results["epoch_val_losses"].append(avg_val_loss)
        config_results["epoch_map50"].append(map50)
        config_results["epoch_map50_95"].append(map50_95)

# ---- SUMMARY ----
print("\n===== RESULTS SUMMARY =====")
for i, res in enumerate(all_results):
    print(
        f"Config {i+1} {res['config']}\n"
        f"  Final Loss:     {res['epoch_losses'][-1]:.4f}\n"
        f"  Best mAP@50:    {max(res['epoch_map50']):.4f}\n"
        f"  Best mAP@50:95: {max(res['epoch_map50_95']):.4f}\n"
    )

with open("all_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

In [ ]:
import time

SAVE_DIR   = "/kaggle/working/frcnn_output"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_CFG = {"lr": 5e-4, "momentum": 0.95, "weight_decay": 1e-4}
EPOCHS   = 10
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── loaders ───────────────────────────────────────────────────────────────────
train_loader = DataLoader(COCOSubsetDataset(TRAIN_JSON, IMAGE_DIR), batch_size=4,
                          shuffle=True,  collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader   = DataLoader(COCOSubsetDataset(VAL_JSON,   IMAGE_DIR), batch_size=4,
                          shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)
test_loader  = DataLoader(COCOSubsetDataset(TEST_JSON,  IMAGE_DIR), batch_size=1,
                          shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)

# ── train ─────────────────────────────────────────────────────────────────────
model = get_model(num_classes=6).to(device)
optimizer = torch.optim.SGD(model.parameters(), **BEST_CFG)
scaler = torch.amp.GradScaler("cuda")

history = {"epoch_train_loss": [], "epoch_val_loss": [], "epoch_map50": [], "epoch_map50_95": []}
start = time.time()

for epoch in range(EPOCHS):
    # train
    model.train()
    epoch_loss = 0
    for imgs, targets in tqdm(train_loader, desc=f"Train {epoch+1}/{EPOCHS}"):
        imgs    = [i.to(device) for i in imgs]
        targets = [{k: v.to(device) for k,v in t.items()} for t in targets]
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            loss_dict = model(imgs, targets)
            loss = sum(loss_dict.values())
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    avg_train_loss = epoch_loss / len(train_loader)

    # val loss + mAP
    metric = MeanAveragePrecision(iou_type="bbox")
    val_loss = 0
    with torch.no_grad():
        for imgs, targets in tqdm(val_loader, desc=f"Val   {epoch+1}/{EPOCHS}", leave=False):
            imgs    = [i.to(device) for i in imgs]
            targets = [{k: v.to(device) for k,v in t.items()} for t in targets]
            model.train()
            with torch.amp.autocast("cuda"):
                val_loss += sum(model(imgs, targets).values()).item()
            model.eval()
            with torch.amp.autocast("cuda"):
                preds = model(imgs)
            metric.update(
                [{k: v.cpu() for k,v in p.items()} for p in preds],
                [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in targets]
            )
    avg_val_loss = val_loss / len(val_loader)
    scores = metric.compute()
    map50, map5095 = scores["map_50"].item(), scores["map"].item()

    history["epoch_train_loss"].append(avg_train_loss)
    history["epoch_val_loss"].append(avg_val_loss)
    history["epoch_map50"].append(map50)
    history["epoch_map50_95"].append(map5095)
    print(f"Epoch {epoch+1} | train_loss={avg_train_loss:.4f} val_loss={avg_val_loss:.4f} mAP@50={map50:.4f} mAP@50:95={map5095:.4f}")

history["training_time_seconds"] = time.time() - start
torch.save(model.state_dict(), f"{SAVE_DIR}/frcnn_best.pt")
with open(f"{SAVE_DIR}/frcnn_history.json", "w") as f:
    json.dump(history, f, indent=2)
print("Checkpoint and history saved.")

# ── test ──────────────────────────────────────────────────────────────────────
model.eval()
metric = MeanAveragePrecision(iou_type="bbox")
with torch.no_grad():
    for imgs, targets in tqdm(test_loader, desc="Testing"):
        imgs = [i.to(device) for i in imgs]
        with torch.amp.autocast("cuda"):
            preds = model(imgs)
        metric.update(
            [{k: v.cpu() for k,v in p.items()} for p in preds],
            [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in targets]
        )

test_scores = metric.compute()
test_results = {
    "mAP50":    test_scores["map_50"].item(),
    "mAP50_95": test_scores["map"].item(),
    "mAP_per_class": test_scores["map_per_class"].tolist(),
}
print(f"\nTest mAP@50={test_results['mAP50']:.4f} | mAP@50:95={test_results['mAP50_95']:.4f}")
with open(f"{SAVE_DIR}/frcnn_test_results.json", "w") as f:
    json.dump(test_results, f, indent=2)
print("Test results saved.")

## YOLO Implementation

In [ ]:
model = YOLO("yolov8n.pt")
model.tune(
    data="/kaggle/working/dataset.yaml",
    epochs=20,          
    iterations=3,       
    imgsz=640,
    batch=32,
    device=0,           
    workers=4,
    project="/kaggle/working/runs",
    name="yolov8_tune",
)


In [ ]:
SAVE_DIR  = "/kaggle/working/yolo_output"
os.makedirs(SAVE_DIR, exist_ok=True)

# point this at your best tune checkpoint
MODEL_PATH = "/kaggle/working/runs/yolov8_tune/weights/best.pt"

model = YOLO(MODEL_PATH)

results = model.val(
    data="/kaggle/working/dataset.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    save_json=True,
    project=SAVE_DIR,
    name="test_eval",
)

test_results = {
    "mAP50":         results.box.map50,
    "mAP50_95":      results.box.map,
    "mAP_per_class": results.box.maps.tolist(),
    "class_names":   ["person", "car", "bicycle", "dog", "bus"],
}
print(f"Test mAP@50={test_results['mAP50']:.4f} | mAP@50:95={test_results['mAP50_95']:.4f}")
with open(f"{SAVE_DIR}/yolo_test_results.json", "w") as f:
    json.dump(test_results, f, indent=2)
print("Saved to", SAVE_DIR)


In [ ]:
model = YOLO("yolov8n.pt")
model.train(
    data="/kaggle/working/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=32,        # can push higher now that you have 2x T4
    device=[0, 1],
    workers=4,
    project="/kaggle/working/runs",
    name="yolov8_5class",
    patience=10,
)

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/cv_project_data", "zip", "/kaggle/working")


In [ ]:
from ultralytics.utils import SETTINGS
SETTINGS["raytune"] = False
model = YOLO("yolov8n.pt")
model.train(
    data="/kaggle/working/dataset.yaml",
    epochs=2,
    imgsz=640,
    batch=16,
    device=[0,1],
    workers=4,
    project="/kaggle/working/runs",
    name="debug_run",
    fraction=0.02,   # only uses 2% of the dataset
)

In [ ]:
import subprocess
result = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True)
print(result.stdout)

In [ ]:
with open("/kaggle/working/labels/train/000000007913.txt") as f:
    print(f.read())

In [ ]:
import shutil, os

dirs_to_delete = [
    "/kaggle/working/labels",
    "/kaggle/working/images",
    "/kaggle/working/coco_converted-2",
    "/kaggle/working/annotations",
    "/kaggle/working/runs",
    "/kaggle/working/train.json",
    "/kaggle/working/val.json",
    "/kaggle/working/test.json",
    "/kaggle/working/dataset.yaml",
]

for path in dirs_to_delete:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"Deleted dir:  {path}")
    elif os.path.isfile(path):
        os.remove(path)
        print(f"Deleted file: {path}")
    else:
        print(f"Not found:    {path}")

print("Done")